## 1. Setup and Data Loading
Here, we import the necessary libraries (`pandas`, `os`) and load the raw dataset. We also verify the data schema to ensure the `body` column exists for processing.

In [11]:
import pandas as pd
import os

file_path = '/Users/nitishrmaladakar/Desktop/infosys/infosys-langgraph-email-assistant-group2/data/sample_emails_with_triage_200.csv' 

# 2. Load the data
try:
    df = pd.read_csv(file_path)
    print("Data loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Please check the filename.")

Data loaded successfully!


,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [12]:
df = pd.read_csv(r"/Users/nitishrmaladakar/Desktop/infosys/infosys-langgraph-email-assistant-group2/data/sample_emails_with_triage_200.csv")
print(df.head(50))

    id                sender                      subject  \
0    1       alerts@bank.com       Password Reset Request   
1    2       alerts@bank.com  Congratulations! You've Won   
2    3  no-reply@service.com          Promotion: Big Sale   
3    4        sales@shop.com               Monthly Report   
4    5  no-reply@service.com                       Survey   
5    6       alerts@bank.com        Unusual Login Attempt   
6    7        hr@company.com               Project Update   
7    8        sales@shop.com               Project Update   
8    9     security@bank.com            Account Suspended   
9   10      boss@company.com               Monthly Report   
10  11     support@cloud.com                       Survey   
11  12     news@techblog.com                Policy Update   
12  13  teamlead@company.com         Subscription Renewal   
13  14       alerts@bank.com                       Survey   
14  15     security@bank.com               Security Alert   
15  16     news@techblog

## 2. Text Preprocessing
To prepare the data for the agent, we must reduce noise. This step implements a `clean_email_text` function that:
* **Normalizes Case:** Converts text to lowercase to ensure consistency (e.g., 'Urgent' == 'urgent').
* **Removes Noise:** Strips newlines and extra whitespace.
* **Removes Artifacts:** Uses Regex (`re`) to remove special characters, retaining only alphanumeric text.

**Why this matters:** Clean text ensures that our future LLM and rule-based systems process tokens efficiently without being confused by formatting symbols.

In [13]:
import re

def clean_email_text(text):
    if not isinstance(text, str):
        return ""
    

    text = text.lower()
    

    text = text.replace('\n', ' ').strip()
    

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    return text


df['Clean_text'] = df['body'].apply(clean_email_text)


print("Clean_text column created successfully!")
display(df[['body', 'Clean_text']].head())

Clean_text column created successfully!


,body,Clean_text
0,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at 10...
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr 2551509 is due on 20251212...
2,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at 11...
3,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...
4,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...


## 3. Rule-Based Triage (Baseline Classifier)
We implement a deterministic classifier to act as a **performance baseline** and a "fast-path" router for the agent.

**Logic:**
The system scans the clean text against predefined keyword dictionaries:
1.  **High Priority (`notify_human`):** Checks for critical terms (e.g., 'urgent', 'error'). This is checked *first* for safety.
2.  **Ignore (`ignore`):** Checks for low-value terms (e.g., 'newsletter', 'promotion').
3.  **Operational (`respond_or_act`):** If no specific keywords are found, the email is categorized as a standard request.

In [14]:
def rule_based_triage(text):
    if not isinstance(text, str):
        return "ignore"

    urgent_keywords = ['urgent', 'error', 'fail', 'critical', 'immediately', 'escalate']
    for word in urgent_keywords:
        if word in text:
            return "notify_human"
    

    ignore_keywords = ['unsubscribe', 'newsletter', 'promotion', 'marketing', 'spam', 'ad']
    for word in ignore_keywords:
        if word in text:
            return "ignore"
    

    return "respond_or_act"


df['triage_prediction'] = df['Clean_text'].apply(rule_based_triage)


print(df['triage_prediction'].value_counts())
display(df[['Clean_text', 'triage_prediction']].head())

triage_prediction
respond_or_act    143
notify_human       38
ignore             19
Name: count, dtype: int64


,Clean_text,triage_prediction
0,reminder the client meeting is scheduled at 10...,respond_or_act
1,your invoice of inr 2551509 is due on 20251212...,respond_or_act
2,reminder the client meeting is scheduled at 11...,respond_or_act
3,hello team please find the attached weekly rep...,respond_or_act
4,hello team please find the attached weekly rep...,respond_or_act


## 4. Export Results
Finally, we save the processed dataframe with the new `Clean_text` and `triage_prediction` columns. This CSV will serve as the input for the next phase of the LangGraph development.

In [22]:

output_path = '../data/milestone1_Nitish.csv'


df.to_csv(output_path, index=False)

print(f"File saved successfully at: {output_path}")

File saved successfully at: ../data/milestone1_Nitish.csv


### **Triage Action Logic**
Maps model outputs to system tasks:
* **Human Review:** Escalates urgent/complex alerts.
* **Auto Reply:** Triggers automated responses.
* **Discard:** Filters out spam and noise.

In [15]:
cols = ["id", "sender", "subject", "body", "priority", "triage_label", "Clean_text", "triage_prediction"]
df = pd.DataFrame(df, columns=cols)

def map_action(prediction):
    if prediction == "notify_human":
        return "Escalate to human support"
    elif prediction == "respond":
        return "Send automated response"
    elif prediction == "ignore":
        return "No action required"
    else:
        return "Manual review needed"

df["ideal_response"] = df["triage_prediction"].apply(map_action)
df[["triage_prediction", "ideal_response"]].head()

,triage_prediction,ideal_response
0,respond_or_act,Manual review needed
1,respond_or_act,Manual review needed
2,respond_or_act,Manual review needed
3,respond_or_act,Manual review needed
4,respond_or_act,Manual review needed
